# 🚀 KritiAi: Fine-Tuning DeepSeek-R1-Distill-Qwen-7B with Unsloth
### Autonomous AI Assistant & Reasoning Engine

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atultiwari997721/KritiAi_App/blob/main/custom_model_kritiai/colab/KritiAi_DeepSeek_R1_FineTuning.ipynb)

> **Important:** Make sure your Colab runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

## 1. Install Unsloth, PyTorch, and Dependencies (Optimized for Free T4)

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps triton
!pip install torchvision datasets huggingface_hub

## 2. Load Base Model with 4-bit Quantization (Unsloth 2x Faster)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detection (Float16 for Tesla T4, Bfloat16 for Ampere+)
load_in_4bit = True # 4-bit quantization reduces 7B VRAM to ~5.5 GB

print("📥 Loading DeepSeek-R1-Distill-Qwen-7B base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Qwen-7B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Base Model Loaded successfully!")

## 3. Attach LoRA Adapters (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ LoRA Adapters configured!")

## 4. Prepare Dataset with DeepSeek-R1 ChatML & Reasoning Template

In [ ]:
from datasets import Dataset

sample_data = [
    {
        "instruction": "Who are you and what are your capabilities?",
        "reasoning": "The user is inquiring about my identity and capabilities. I should introduce myself as KritiAi, highlighting my reasoning, coding, and autonomous capabilities.",
        "response": "I am **KritiAi**, an advanced autonomous AI assistant built with DeepSeek-R1 reasoning architecture. I specialize in deep multi-step logic, full-stack software development, automated system operations, and creative problem solving."
    },
    {
        "instruction": "Write an optimized Python function to compute Fibonacci numbers using dynamic programming memoization.",
        "reasoning": "1. Understand problem: Efficiently calculate the n-th Fibonacci number.\n2. Choose strategy: Top-down memoization dictionary to prevent O(2^n) exponential complexity.\n3. Validate inputs: Non-negative integers.\n4. Write clean code with type annotations.",
        "response": "Here is an optimized Python implementation using memoization:\n\n```python\nfrom typing import Dict\n\ndef fibonacci_memo(n: int, memo: Dict[int, int] = None) -> int:\n    if memo is None:\n        memo = {0: 0, 1: 1}\n    if n < 0:\n        raise ValueError(\"n must be a non-negative integer.\")\n    if n in memo:\n        return memo[n]\n    memo[n] = fibonacci_memo(n - 1, memo) + fibonacci_memo(n - 2, memo)\n    return memo[n]\n\nif __name__ == '__main__':\n    print([fibonacci_memo(i) for i in range(10)])\n```"
    }
]

def format_prompts(batch):
    formatted_texts = []
    for inst, reason, resp in zip(batch["instruction"], batch["reasoning"], batch["response"]):
        text = f"<｜User｜>{inst}<｜Assistant｜><think>\n{reason}\n</think>\n{resp}<｜end of sentence｜>"
        formatted_texts.append(text)
    return {"text": formatted_texts}

raw_dataset = Dataset.from_list(sample_data)
train_dataset = raw_dataset.map(format_prompts, batched=True)
print(f"📊 Prepared {len(train_dataset)} training samples.")

## 5. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

print("🚀 Starting Training on Tesla T4...")
trainer_stats = trainer.train()
print("🎉 Training Completed Successfully!")

## 6. Test Inference

In [ ]:
FastLanguageModel.for_inference(model)

prompt = "<｜User｜>Who are you?<｜Assistant｜><think>\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])

## 7. Save & Push LoRA Adapters to Hugging Face Hub

In [ ]:
# Paste your Hugging Face credentials here:
HF_USERNAME = "your-hf-username"
HF_REPO_NAME = f"{HF_USERNAME}/KritiAi"
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx" # Your HF Write Token

model.save_pretrained("KritiAi_LoRA")
tokenizer.save_pretrained("KritiAi_LoRA")

print(f"☁️ Uploading to Hugging Face Hub: {HF_REPO_NAME}...")
model.push_to_hub(HF_REPO_NAME, token = HF_TOKEN)
tokenizer.push_to_hub(HF_REPO_NAME, token = HF_TOKEN)
print(f"✅ Successfully Published at: https://huggingface.co/{HF_REPO_NAME}")